# Train Net Line Pose Model

This notebook trains a one-class YOLO pose/keypoint model from datasets exported by `tools/export_yolo/export_net_pose_dataset.py`.

Expected input after export:

```txt
data/exports/net_yolo_pose/
  dataset.yaml
  images/train/*.jpg
  images/val/*.jpg
  labels/train/*.txt
  labels/val/*.txt
```

The model predicts one `net` object with three keypoints: left endpoint, middle/dip, and right endpoint. This matches the labeler net-line format more directly than thin segmentation masks.


## 1. Runtime Check

In Colab, choose `Runtime > Change runtime type > GPU` before training.


In [ ]:
import sys
import platform

print("python", sys.version)
print("platform", platform.platform())

try:
    import torch
    print("torch", torch.__version__)
    print("cuda available", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu", torch.cuda.get_device_name(0))
except ImportError:
    print("torch not installed yet")


## 2. Install Training Dependencies

In [ ]:
%pip install -q ultralytics

## 3. Provide Dataset

Run the next cell in Colab and choose `net_yolo_pose.zip` when the picker appears. If `net_yolo_pose.zip` already exists in `/content`, the cell will reuse it without opening the picker.

Locally, create it after export with:

```bash
cd data/exports
zip -r net_yolo_pose.zip net_yolo_pose
```


In [ ]:
from IPython.display import clear_output
clear_output(wait=False)
print("Upload output reset. Now run the next cell.")


In [ ]:
from pathlib import Path
import shutil
import zipfile
from IPython.display import clear_output

clear_output(wait=False)

DATASET_ZIP = "net_yolo_pose.zip"
extract_dir = Path("/content/net_yolo_pose")

def find_existing_zip():
    candidates = [Path(DATASET_ZIP), Path("/content") / DATASET_ZIP]
    candidates.extend(Path("/content").glob("net_yolo_pose*.zip"))
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None

zip_path = find_existing_zip()
if zip_path is None:
    try:
        from google.colab import files
    except ModuleNotFoundError as exc:
        raise RuntimeError("This upload cell must be run in Google Colab, or net_yolo_pose.zip must already exist in the current directory.") from exc

    print(f"Choose {DATASET_ZIP}. If the button is greyed out, ignore the old output and rerun this cell with the play button.")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file was uploaded. Rerun this cell and choose net_yolo_pose.zip.")

    if DATASET_ZIP in uploaded:
        zip_path = Path(DATASET_ZIP)
    else:
        zip_names = [name for name in uploaded if name.endswith(".zip")]
        if len(zip_names) != 1:
            raise FileNotFoundError(f"Expected {DATASET_ZIP} or exactly one .zip file, got: {list(uploaded)}")
        zip_path = Path(zip_names[0])
        print(f"Using uploaded zip: {zip_path}")

if extract_dir.exists():
    shutil.rmtree(extract_dir)
extract_dir.mkdir(parents=True, exist_ok=True)

print("using zip:", zip_path.resolve())
print("zip size MB:", round(zip_path.stat().st_size / 1024 / 1024, 2))

with zipfile.ZipFile(zip_path) as archive:
    members = archive.infolist()
    total = len(members)
    print(f"extracting {total} files...")
    for index, member in enumerate(members, start=1):
        archive.extract(member, extract_dir)
        if index == total or index % 10 == 0:
            clear_output(wait=True)
            print("using zip:", zip_path.resolve())
            print("zip size MB:", round(zip_path.stat().st_size / 1024 / 1024, 2))
            print(f"extracting files: {index}/{total}")

matches = sorted(extract_dir.rglob("dataset.yaml"))
if not matches:
    found = sorted(str(path.relative_to(extract_dir)) for path in extract_dir.rglob("*") if path.is_file())[:20]
    raise FileNotFoundError(f"No dataset.yaml found after extracting {zip_path}. First extracted files: {found}")

DATASET_YAML = str(matches[0])
dataset_root = Path(DATASET_YAML).parent.resolve()
Path(DATASET_YAML).write_text(
    f"path: {dataset_root.as_posix()}
"
    "train: images/train
"
    "val: images/val
"
    "kpt_shape: [3, 3]
"
    "flip_idx: [2, 1, 0]
"
    "names:
"
    "  0: net
",
    encoding="utf-8",
)

print("dataset yaml:", DATASET_YAML)
print("dataset root:", dataset_root)
print(Path(DATASET_YAML).read_text())


## 4. Inspect Dataset Size

In [ ]:
dataset_root = Path(DATASET_YAML).parent
total_images = 0
total_labels = 0
for split in ("train", "val"):
    image_count = len(list((dataset_root / "images" / split).glob("*")))
    label_count = len(list((dataset_root / "labels" / split).glob("*.txt")))
    total_images += image_count
    total_labels += label_count
    print(split, "images", image_count, "labels", label_count)

if total_images == 0 or total_labels == 0:
    raise RuntimeError(f"Dataset appears empty at {dataset_root}. Re-upload data/exports/net_yolo_pose.zip and rerun the dataset cell.")


## 5. Train Baseline

Start with the nano pose model. This is a smoke test to see whether direct line/keypoint prediction beats the thin-mask segmentation attempt.


In [ ]:
from ultralytics import YOLO

dataset_root = Path(DATASET_YAML).parent.resolve()
Path(DATASET_YAML).write_text(
    f"path: {dataset_root.as_posix()}
"
    "train: images/train
"
    "val: images/val
"
    "kpt_shape: [3, 3]
"
    "flip_idx: [2, 1, 0]
"
    "names:
"
    "  0: net
",
    encoding="utf-8",
)
print(Path(DATASET_YAML).read_text())

model = YOLO("yolo26n-pose.pt")
results = model.train(
    data=DATASET_YAML,
    epochs=30,
    imgsz=640,
    batch=4,
    workers=2,
    cache=False,
    patience=10,
    project="/content/runs/ar_ping_pong",
    name="net_yolo26n_pose_img640",
)


## 6. Validate And Predict

In [ ]:
best = "/content/runs/ar_ping_pong/net_yolo26n_pose_img640/weights/best.pt"
model = YOLO(best)
metrics = model.val(data=DATASET_YAML, imgsz=640)
print(metrics)

prediction_source = dataset_root / "images" / "val"
if not any(prediction_source.glob("*")):
    prediction_source = dataset_root / "images" / "train"

predictions = model.predict(
    source=str(prediction_source),
    imgsz=640,
    conf=0.1,
    save=True,
    project="/content/runs/ar_ping_pong",
    name="net_pose_predictions",
)
print("prediction images:", "/content/runs/ar_ping_pong/net_pose_predictions")


## 7. Save Checkpoint

Download this file after training:

```txt
/content/runs/ar_ping_pong/net_yolo26n_pose_img640/weights/best.pt
```

Save it in the repo as:

```txt
models/net_pose/net_yolo26n_pose_img640.pt
```
